# Building a model
## Building through individual reactions
We begin by constructing a kinetic model for a metabolic system, which consists of biochemical reactions governed by user-defined rate laws.

First, initialize an empty model:

In [1]:
from robustnet import Model

model = Model('demo')
print(model)

WARNING (pytensor.tensor.blas): Using NumPy C-API based implementation for BLAS functions.


Model demo empty


Then a reaction can be created and added to the model by specifying reaction ID, catalytic enzyme ID (None for non-enzymatic reations), substrate and product stoichiometry, as well as kinetic rate expression.

Below is an example of adding the glucose phosphotransferase system reaction using a generalized reversible Hill-type rate law:

<center> glucose (GLC) + phosphoenolpyruvate (PEP) <-> glucose 6-phosphate (G6P) + pyruvate (PYR) </center>

In [2]:
model.add_reaction(
    'GlcPTS',
    enzyme='GlcPTS',
    substrates={'GLC': 1, 'PEP': 1},
    products={'G6P': 1, 'PYR': 1},
    rate_expression='kcat_vGlcPTS * GlcPTS * '
                    '(GLC * PEP / (Km_GLC_vGlcPTS * Km_PEP_vGlcPTS)) * '
                    '(1 - G6P * PYR / (GLC * PEP) / Keq_vGlcPTS) / '
                    '((1 + PEP / Km_PEP_vGlcPTS + PYR / Km_PYR_vGlcPTS) * '
                    '(1 + GLC / Km_GLC_vGlcPTS + G6P / Km_G6P_vGlcPTS))'
)

Rate expressions can be provided as plain strings. Metabolite variables, enzyme variables, and kinetic parameters are automatically parsed from the expression. 

Kinetic parameter names can be arbitrary, although the following naming are recommended for clarity:
- catalytic constants: "kcat_{reaction_id}"
- Michaelis constants: "Km_{metabolite_id}\_{reaction_id}"
- equilibrium constants: "Keq_{reaction_id}"

<div class="alert alert-info">
<b>Note:</b> <br></br> 
Metabolite IDs cannot start with a number and must not overlap with enzyme IDs.
Enzyme and metabolite identifiers should remain consistent with those used in the rate expressions.
</div>

Next, we we add the reaction catalyzed by glucose-6-phosphate isomerase:

<center> glucose 6-phosphate (G6P) <-> fructose 6-phosphate (F6P) </center>

using a reversible Michaelis–Menten rate law with competitive inhibition by PEP and mixed inhibition by 6-phospho-D-gluconate (PGlc):

In [3]:
model.add_reaction(
    'PGI',
    enzyme='PGI',
    substrates={'G6P': 1},
    products={'F6P': 1},
    rate_expression='kcat_vPGI * PGI * '
                    '(G6P / Km_G6P_vPGI) * '
                    '(1 - F6P / G6P / Keq_vPGI) / '
                    '(1 + G6P / Km_G6P_vPGI + F6P / Km_F6P_vPGI + PEP/Ki_PEP_vPGI + PGlc/Ki_PGlc_vPGI) * '
                    '(Ki_PGlc_vPGI / (Ki_PGlc_vPGI + PGlc))'
)
print(model)

Model demo with 2 reactions and 5 metabolites


The parsed kinetic parameters from all added reactions can then be inspected as follows:

In [4]:
model.parsed_kinetic_parameters()

{'GlcPTS': ['Keq_vGlcPTS',
  'Km_G6P_vGlcPTS',
  'Km_GLC_vGlcPTS',
  'Km_PEP_vGlcPTS',
  'Km_PYR_vGlcPTS',
  'kcat_vGlcPTS'],
 'PGI': ['Keq_vPGI',
  'Ki_PEP_vPGI',
  'Ki_PGlc_vPGI',
  'Km_F6P_vPGI',
  'Km_G6P_vPGI',
  'PGlc',
  'kcat_vPGI']}

<div class="alert alert-info">
<b>Note:</b> <br></br>
In the PGI reaction, "PGlc" interpreted as a kinetic parameter because it appears only in the rate expression. Once it is introduced as a substrate or product in another reaction, it will instead be treated as a metabolite variable.
</div>

More generally, after all reactions are added, any variable that appears only in rate expressions but not in the `substrates`, `products`, or `enzyme` argument, is automatically classified as a kinetic parameter.

To remove reaction(s) from model, use `remove_reaction` method:

In [5]:
model.remove_reaction(['PGI'])
print(model)

Model demo with 1 reactions and 4 metabolites


## Reading from file
Alternatively, an entire model can be loaded directly from a file containing the following required fields: "Reaction", "Enzyme", "Substrates", "Products" and "Rate expression". 

An example [*E .coli* model](https://github.com/Chaowu88/robustnet/blob/main/models/e_coli/e_coli_model.xlsx) is available in the repository for reference.

In [6]:
MODEL_FILE = '../../models/e_coli/e_coli_model.xlsx'

model = Model('ecoli')
model.read_from_file(MODEL_FILE)
print(model)

Model ecoli with 64 reactions and 50 metabolites


The reaction information stored in the model can then be inspected as follows (showing the first five reactions):

In [7]:
for rxn_id, rxn_info in list(model.reaction_info.items())[:5]:
    print(rxn_id, '\n', rxn_info)

GlcPTS 
 Reaction(enzyme='GlcPTS', substrates={'GLC': 1.0, 'PEP': 1.0}, products={'G6P': 1.0, 'PYR': 1.0}, ratelaw=Ratelaw(enz='GlcPTS', metabs=['G6P', 'GLC', 'PEP', 'PYR'], kparams=['Keq_vGlcPTS', 'Km_G6P_vGlcPTS', 'Km_GLC_vGlcPTS', 'Km_PEP_vGlcPTS', 'Km_PYR_vGlcPTS', 'kcat_vGlcPTS'], expr=GLC*GlcPTS*PEP*kcat_vGlcPTS*(-G6P*PYR/(GLC*Keq_vGlcPTS*PEP) + 1)/(Km_GLC_vGlcPTS*Km_PEP_vGlcPTS*(1 + PYR/Km_PYR_vGlcPTS + PEP/Km_PEP_vGlcPTS)*(G6P/Km_G6P_vGlcPTS + GLC/Km_GLC_vGlcPTS + 1))))
PGI 
 Reaction(enzyme='PGI', substrates={'G6P': 1.0}, products={'F6P': 1.0}, ratelaw=Ratelaw(enz='PGI', metabs=['F6P', 'G6P', 'PEP', 'PGlc'], kparams=['Keq_vPGI', 'Ki_PEP_vPGI', 'Ki_PGlc_vPGI', 'Km_F6P_vPGI', 'Km_G6P_vPGI', 'kcat_vPGI'], expr=G6P*Ki_PGlc_vPGI*PGI*kcat_vPGI*(-F6P/(G6P*Keq_vPGI) + 1)/(Km_G6P_vPGI*(Ki_PGlc_vPGI + PGlc)*(F6P/Km_F6P_vPGI + G6P/Km_G6P_vPGI + 1 + PGlc/Ki_PGlc_vPGI + PEP/Ki_PEP_vPGI))))
PFK 
 Reaction(enzyme='PFK', substrates={'ATP': 1.0, 'F6P': 1.0}, products={'ADP': 1.0, 'FBP': 1.0}, 

PGlc now recognized as a metabolite in the model.

In [8]:
model.reaction_info['PGL'].products

{'PGlc': 1.0}

In [9]:
model.parsed_kinetic_parameters('PGI')

{'PGI': ['Keq_vPGI',
  'Ki_PEP_vPGI',
  'Ki_PGlc_vPGI',
  'Km_F6P_vPGI',
  'Km_G6P_vPGI',
  'kcat_vPGI']}